# Setup

## Imports

In [ ]:
# Native packages
from pathlib import Path
from importlib import reload
import re
from collections.abc import Iterable

# Other packages
import pandas as pd
import xarray as xr
import pint_xarray
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.colors as colors
import matplotlib as mpl
from IPython.display import HTML, display, Image
from tqdm.notebook import tqdm
import pickle as pkl
from xhistogram.xarray import histogram

from metpy.units import units

# My packages
import xarrams as xrr
import ps
import ps.projects.dc_entrainment as dc
import carlee_tools as ct
import skyvista as sv

# Alias for first value
fv = ct.first_value

plt.style.use("cd_slides")

## Parameters

In [ ]:
EXPERIMENT_GROUP_NAME = "ic"
EXPERIMENT_SUBGROUP_NAME = "ic_p20b"

EXPERIMENT_NAMES = [
    "bubbleamp-2_qv0-11_nparcels-1e7",
]

## Dask cluster setup

In [ ]:
client = ps.backends.dask.connect_or_spawn_cluster()

## Function: create figure for all experiments

In [ ]:
def subplots_for_datasets(
    datasets,
    ncols=None,
    nrows=None,
    width_per_axes=3,
    height_per_axes=3,
    width_padding=0,
    height_padding=1,
    **kwargs,
):
    n_datasets = len(datasets)

    if not nrows and not ncols:
        nrows = np.ceil(np.sqrt(n_datasets))
        ncols = nrows
        if nrows * (ncols - 1) >= n_datasets:
            # If a square is overly large, prefer removing columns
            nrows -= 1
    elif nrows and not ncols:
        ncols = np.ceil(n_datasets / nrows)
    elif ncols and not nrows:
        nrows = np.ceil(n_datasets / ncols)

    nrows = int(nrows)
    ncols = int(ncols)

    figsize = (
        width_per_axes * ncols + width_padding,
        height_per_axes * nrows + height_padding,
    )
    fig, axs = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=figsize, squeeze=False, **kwargs
    )
    # Hide unnecessary axes
    for ax in axs.flatten()[n_datasets:]:
        ax.axis("off")
    axs = axs[:n_datasets]
    # Set axes titles
    for ix, exp in enumerate(datasets.keys()):
        axs.flatten()[ix].set_title(exp)
    return fig, axs

## Activity function

In [ ]:
def A(ds):
    # "Activity operator" following Yeo and Romps 2015; really just an in-updraft flag
    return (ds["WC"] >= 1) & (ds["RCP"] >= 1e-5)

# Read data

## CM1

In [ ]:
raw_cm1_dss = {}
cm1_dss = {}

for experiment in EXPERIMENT_NAMES:
    output_dir = (
        dc.PROJECT_DATA_DIR
        / EXPERIMENT_GROUP_NAME
        / EXPERIMENT_SUBGROUP_NAME
        / experiment
        / "output"
    )
    print(output_dir)
    ds = xrr.read_cm1_output(sorted(list(output_dir.glob("cm1out_0*.nc"))))
    raw_cm1_dss[experiment] = ds
    # Make it rams-like
    ds = xrr.cm1.coerce_to_ramslike(ds)
    # ds = xrr.calculate_derived_variables(ds)
    ds = xrr.calculate_thermodynamic_variables(ds)
    ds = xrr.calculate_bsr_variables(ds, base_state=ds.isel(time=0))
    # Calculate activity operator
    ds["active"] = A(ds)
    cm1_dss[experiment] = ds

fv(cm1_dss)

## Parcels

In [ ]:
reload(xrr.cm1)
reload(xrr.calculations)
reload(xrr)


def preprocess_parcel_ds(ds):
    # Handle dimension renaming and dropping unused
    ds = ds.squeeze().rename({"xh": "parcel_ix"}).drop_vars(["yh", "zh"])
    # Make ramslike
    ds = xrr.cm1.coerce_to_ramslike(ds, keep_non_coercable=True)
    # u/v/w are assumed to be called uinterp/vinterp/winterp, so rename them
    # manually for parcels
    ds = ds.rename({x: f"{x.upper()}C" for x in ["u", "v", "w"]})
    # Calculate derived thermodynamic variables
    ds = xrr.calculate_thermodynamic_variables(ds)
    # Calculate base-state relative to get t_minutes
    ds = xrr.calculate_bsr_variables(ds, base_state=ds.isel(time=0), bsr_variables=[])
    # Add activity operator
    ds["active"] = A(ds).astype(int)
    # Calculate its change at each timestep to get entrainment/detrainment operators
    # Compute change in the activity operator between consecutive timesteps.
    dA = ds["active"].diff("time")

    # Precalculate origin position
    for var in ["x", "y", "z"]:
        ds[f"{var}_origin"] = ds.isel(time=0)[var]

    # Create a zero-filled DataArray for the first time index so dA has the
    # same length as ds["active"]. This represents "no change" before the first step.
    zero_first = xr.zeros_like(ds["active"].isel(time=0)).expand_dims(
        time=[ds["time"].isel(time=0).values.item()]
    )

    # Prepend the zero first-step array to the diffed array so indices align.
    dA = xr.concat([zero_first, dA], dim="time")
    ds["entraining"] = dA == 1
    ds["detraining"] = dA == -1

    # Do some checks
    assert ds["entraining"].isel(time=0).sum().values.item() == 0
    assert ds["detraining"].isel(time=0).sum().values.item() == 0
    assert ds["active"].isel(time=0).sum().values.item() == 0

    return ds


raw_parcels_dss = {
    experiment: xr.open_dataset(
        dc.PROJECT_DATA_DIR
        / EXPERIMENT_GROUP_NAME
        / EXPERIMENT_SUBGROUP_NAME
        / experiment
        / "output"
        / "cm1out_pdata.nc",
        engine="h5netcdf",
        chunks="auto",
    )
    for experiment in EXPERIMENT_NAMES
}

parcels_dss = {k: preprocess_parcel_ds(ds) for k, ds in raw_parcels_dss.items()}

small_parcels_dss = {
    k: ds.sel(parcel_ix=ct.maybe_random_choice(ds["parcel_ix"].values, 250))
    for k, ds in parcels_dss.items()
}

fv(parcels_dss)

# Postprocessing

## Eulerian entrainment from parcels

In [ ]:
def eulerian_entrainment(parcel_ds, cm1_ds):
    xyz_dims = ["x", "y", "z"]
    cm1_ds = cm1_ds.copy()
    # Build bin edges: prepend a leading 0, compute interior midpoints between
    # adjacent grid cell centers, then extrapolate one final edge past the last center.
    # np.append only accepts two arrays; use np.concatenate for three-way joins.
    bin_edges = {
        dim: np.concatenate([
            # Leading edge: use 0 as the lower boundary of the first bin
            np.array([0]),
            # Interior edges: midpoint between each pair of adjacent cell centers
            (cm1_ds[dim] - (cm1_ds[dim].diff(dim) / 2)).values,
            # Trailing edge: extrapolate one half-spacing past the last center
            np.array([
                cm1_ds[dim].values[-1]
                + (cm1_ds[dim].values[-1] - cm1_ds[dim].values[-2]) / 2
            ]),
        ])
        for dim in xyz_dims
    }
    # Per-bin widths along each dim. Length = number of bins (len(edges) - 1).
    bin_widths = {dim: np.diff(bin_edges[dim]) for dim in xyz_dims}
    # Per-bin (x, y, z) volume, broadcast to a 3D array so we can divide the
    # binned histogram by it below to convert mass counts to a mass density.
    bin_volume = (
        bin_widths["x"][:, None, None]
        * bin_widths["y"][None, :, None]
        * bin_widths["z"][None, None, :]
    )
    retvals = {}
    for direction in ["entrain", "detrain"]:
        # Histogram of (de)trained parcel mass into the (x, y, z) grid cell bins.
        binned_mass = histogram(
            *(parcel_ds[dim] for dim in xyz_dims),
            bins=list(bin_edges.values()),
            dim=["parcel_ix"],
            weights=parcel_ds[f"{direction}ing"],
        ).rename({f"{dim}_bin": dim for dim in xyz_dims})
        # Divide by the (x, y, z) bin volume to convert binned mass into a
        # volumetric (de)training density with units of [weights] / [x y z].
        retvals[direction] = binned_mass / bin_volume
    return list(retvals.values())


for exp, cm1_ds in cm1_dss.items():
    e, d = eulerian_entrainment(parcel_ds=parcels_dss[exp], cm1_ds=cm1_ds)
    # cm1_ds["M_entraining"], cm1_ds["M_detraining"] = eulerian_entrainment(
    #     parcel_ds=parcels_dss[exp], cm1_ds=cm1_ds
    # )
    # cm1_dss[exp] = cm1_ds

# display(fv(cm1_dss))
e

# Checks and diagnostics

## Base state

In [ ]:
reload(xrr.soundings)
reload(xrr)

for exp, cm1_ds in cm1_dss.items():
    xrr.plot_base_state_diagnostics(cm1_ds.isel(time=0))

## Forcing

In [ ]:
def plot_forcing_diagnostics(
    t0_ds,
    forcing_variable_names,
    plan_view_z_index=5,
    cross_section_y_fraction=0.5,
):
    """Per-variable 2x2 diagnostic of the initial (t=0) forcing fields.

    Each variable in `forcing_variable_names` gets its own 2x2 subfigure:
      - Row 0 (the slices):
          col 0 = vertical cross-section in the x-z plane, taken at a fixed
                  y (the domain mid-plane by default);
          col 1 = horizontal plan view in the x-y plane, taken at a fixed
                  model level (z index `plan_view_z_index`).
      - Row 1 (the spread of those slices): the standard deviation of the
          field across the dimension that the row-0 panel above it collapsed.
          col 0 = std over y (the dimension the cross-section fixed);
          col 1 = std over z (the dimension the plan view fixed).
        So each row-1 panel answers "how much does this field vary through
        the slice shown directly above it?".

    Every panel is labelled and carries its own colorbar (units on the
    colorbar). Perturbation variables (those whose name ends in "_bsr") are
    drawn on a symmetric diverging scale so the sign reads clearly; their
    standard-deviation panels are non-negative and so use a sequential scale.
    """
    # Horizontal coordinates in km for readable axes; keep z in metres
    x_km = t0_ds["x"].values / 1000
    y_km = t0_ds["y"].values / 1000
    z_m = t0_ds["z"].values

    # y location of the vertical cross-section (domain mid-plane by default)
    cross_section_y = float(t0_ds["y"].max().values) * cross_section_y_fraction
    # Actual model level used for the plan view (read back for labelling)
    plan_view_z_m = float(t0_ds["z"].isel(z=plan_view_z_index).values)

    # One stacked 2x2 subfigure per requested variable
    n_variables = len(forcing_variable_names)
    fig = plt.figure(figsize=(9, 7 * n_variables), layout="constrained")
    # `np.atleast_1d` so the single-variable case still gives an iterable
    subfigs = np.atleast_1d(fig.subfigures(nrows=n_variables, ncols=1))

    def draw_panel(subfig, ax, x_coords, y_coords, field_2d, units_label, is_diverging):
        """Plot one pcolormesh panel and attach a labelled colorbar."""
        if is_diverging:
            # Symmetric scale about zero so the sign of the perturbation is clear
            field_max_absolute = float(np.nanmax(np.abs(field_2d))) or 1.0
            pc = ax.pcolormesh(
                x_coords,
                y_coords,
                field_2d,
                cmap="RdBu_r",
                norm=colors.TwoSlopeNorm(
                    vcenter=0,
                    vmin=-field_max_absolute,
                    vmax=field_max_absolute,
                ),
            )
        else:
            pc = ax.pcolormesh(x_coords, y_coords, field_2d, cmap="viridis")
        # Units live on the colorbar (this is a pcolormesh, not a line plot)
        subfig.colorbar(pc, ax=ax, label=units_label)
        return pc

    for variable_name, subfig in tqdm(zip(forcing_variable_names, subfigs)):
        # Units for the colorbar labels, if the variable carries them
        units_label = t0_ds[variable_name].attrs.get("units", "")
        # Perturbation fields (theta_bsr etc.) are signed -> diverging colour scale
        is_perturbation = variable_name.endswith("_bsr")

        # --- Row 0: the two slices ---

        # Vertical cross-section at fixed y; orient as (z, x) for pcolormesh
        cross_section_zx = (
            t0_ds[variable_name]
            .sel(y=cross_section_y, method="nearest")
            .transpose("z", "x")
        )
        # Horizontal plan view at a fixed model level; orient as (y, x)
        plan_view_yx = (
            t0_ds[variable_name].isel(z=plan_view_z_index).transpose("y", "x")
        )

        # --- Row 1: spread of each slice across the dimension it collapsed ---

        # Std over y -> how much the field varies across the cross-section plane
        cross_section_std_zx = t0_ds[variable_name].std("y").transpose("z", "x")
        # Std over z -> how much the field varies through the column at the plan view
        plan_view_std_yx = t0_ds[variable_name].std("z").transpose("y", "x")

        # Compute all four panels for this variable together so dask reads
        # the input chunks once rather than four separate times
        panel_dataset = xr.Dataset({
            "cross_section": cross_section_zx,
            "plan_view": plan_view_yx,
            "cross_section_std": cross_section_std_zx,
            "plan_view_std": plan_view_std_yx,
        }).compute()

        axs = subfig.subplots(nrows=2, ncols=2)
        subfig.suptitle(variable_name)

        # Row 0, col 0: cross-section slice
        draw_panel(
            subfig,
            axs[0, 0],
            x_km,
            z_m,
            panel_dataset["cross_section"].values,
            units_label,
            is_diverging=is_perturbation,
        )
        axs[0, 0].set_title(f"x-z cross-section (y ≈ {cross_section_y / 1000:.0f} km)")
        axs[0, 0].set_xlabel("x (km)")
        axs[0, 0].set_ylabel("z (m)")

        # Row 0, col 1: plan-view slice
        draw_panel(
            subfig,
            axs[0, 1],
            x_km,
            y_km,
            panel_dataset["plan_view"].values,
            units_label,
            is_diverging=is_perturbation,
        )
        axs[0, 1].set_title(f"x-y plan view (z ≈ {plan_view_z_m:.0f} m)")
        axs[0, 1].set_xlabel("x (km)")
        axs[0, 1].set_ylabel("y (km)")
        axs[0, 1].set_aspect("equal")

        # Row 1, col 0: std of the cross-section field across y (non-negative)
        draw_panel(
            subfig,
            axs[1, 0],
            x_km,
            z_m,
            panel_dataset["cross_section_std"].values,
            units_label,
            is_diverging=False,
        )
        axs[1, 0].set_title("Std over y")
        axs[1, 0].set_xlabel("x (km)")
        axs[1, 0].set_ylabel("z (m)")

        # Row 1, col 1: std of the plan-view field across z (non-negative)
        draw_panel(
            subfig,
            axs[1, 1],
            x_km,
            y_km,
            panel_dataset["plan_view_std"].values,
            units_label,
            is_diverging=False,
        )
        axs[1, 1].set_title("Std over z")
        axs[1, 1].set_xlabel("x (km)")
        axs[1, 1].set_ylabel("y (km)")
        axs[1, 1].set_aspect("equal")

    return fig


# Variables to diagnose; each gets its own 2x2 subfigure
forcing_variable_names = ["THETA_bsr", "WC", "RV", "RH"]

for exp, cm1_ds in cm1_dss.items():
    t0_ds = cm1_ds.isel(time=0)
    fig = plot_forcing_diagnostics(t0_ds, forcing_variable_names)
    fig.suptitle(exp)

## Global extrema

In [ ]:
def plot_sentinel_time_series(
    ds,
    mass_flux_reference_height_m=3000,
    cloud_condensate_threshold_kg_per_kg=1e-5,
):
    """One line per panel — fast sanity scan for blowups / flatlines.

    Each panel shows a single domain-wide quantity vs simulation time.
    Sentinels span dynamics, thermodynamics, precipitation, and an
    integrated mass-conservation check; if any individual one goes
    non-physical, it stands out without having to scan whole fields.
    """
    # Collect all hydrometeor mixing ratios that this run actually carries
    hydrometeor_mixing_ratio_vars = [
        v
        for v in ["RCP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP"]
        if v in ds.data_vars
    ]
    # Total condensate (cloud + rain + ice + snow + ...) at every grid point
    total_condensate_mixing_ratio = sum(ds[v] for v in hydrometeor_mixing_ratio_vars)
    # Horizontal wind speed magnitude at every grid point
    horizontal_wind_speed = np.sqrt(ds["UC"] ** 2 + ds["VC"] ** 2)

    # --- Per-time scalars (each one is a (time,) array) ---

    # Peak updraft vs peak downdraft kept separately — a strong downdraft
    # surge that doesn't trip the updraft side is still visible this way
    domain_max_w = ds["WC"].max(["x", "y", "z"])
    domain_min_w = ds["WC"].min(["x", "y", "z"])

    # Peak horizontal wind — picks up gust fronts, outflow, low-level jets
    domain_max_horizontal_wind = horizontal_wind_speed.max(["x", "y", "z"])

    # Peak instantaneous surface precip rate
    domain_max_precip_rate = ds["PCPRR_mm_hr"].max(["x", "y"])

    # Total mass of water (kg) that has hit the surface so far over the whole domain
    domain_integrated_accumulated_precip = ds["ACCPR"].integrate(["x", "y"])

    # Cold pool intensity = coldest θ_v perturbation at the lowest model level
    # (more negative = more intense cold pool).
    # `isel(z=0)` keeps z as a scalar coord (= lowest z-level value); drop it
    # so this sentinel can be combined into a Dataset with the other sentinels,
    # which either have no z coord at all or carry a different scalar z value.
    surface_min_theta_v_perturbation = (
        ds["theta_v_bsr"].isel(z=0).min(["x", "y"]).drop_vars("z")
    )

    # Total water mass in the air column (vapor + condensate, kg).
    # Should drift only due to surface flux and precipitation losses;
    # a sudden jump is a strong hint at numerical or coupling trouble.
    total_water_mass = (
        ds["DN0"] * (ds["RV"] + total_condensate_mixing_ratio)
    ).integrate(["x", "y", "z"])

    # Upward convective mass flux at a chosen reference height:
    # ∫ρw dA where w>0. Less spiky than max W; tracks the *size* of
    # the upward branch of convection, not just the peak point.
    slice_at_reference_height = ds.sel(z=mass_flux_reference_height_m, method="nearest")
    # Read the actual z of the selected level before dropping the coord
    actual_reference_height_m = float(slice_at_reference_height["z"].values)
    positive_w_indicator = (slice_at_reference_height["WC"] > 0).astype(float)
    # Drop the scalar z coord (set by `.sel(z=..., method="nearest")`) for the
    # same reason as the cold-pool sentinel above
    upward_mass_flux_at_reference_height = (
        (
            slice_at_reference_height["DN0"]
            * slice_at_reference_height["WC"]
            * positive_w_indicator
        )
        .integrate(["x", "y"])
        .drop_vars("z")
    )

    # Cloud-top height = highest z where total condensate exceeds the threshold,
    # then taken as domain-max over (x, y). Tracks anvil rise / storm depth.
    cloudy_indicator = (
        total_condensate_mixing_ratio > cloud_condensate_threshold_kg_per_kg
    )
    # ds["z"] broadcasts against the indicator; zero at non-cloudy points
    cloudy_heights = ds["z"] * cloudy_indicator
    domain_max_cloud_top_height = cloudy_heights.max(["x", "y", "z"])

    # --- Compute all sentinels at once so dask reads input chunks once ---

    sentinel_series = xr.Dataset({
        "max_w": domain_max_w,
        "min_w": domain_min_w,
        "max_horizontal_wind": domain_max_horizontal_wind,
        "max_precip_rate": domain_max_precip_rate,
        "integrated_accumulated_precip": domain_integrated_accumulated_precip,
        "surface_min_theta_v_perturbation": surface_min_theta_v_perturbation,
        "total_water_mass": total_water_mass,
        "upward_mass_flux": upward_mass_flux_at_reference_height,
        "max_cloud_top_height": domain_max_cloud_top_height,
    }).compute()

    # (panel title, sentinel-series variable name, units label)
    panel_specs = [
        ("Max updraft speed", "max_w", "m/s"),
        ("Min downdraft speed", "min_w", "m/s"),
        ("Max horizontal wind", "max_horizontal_wind", "m/s"),
        ("Max rainfall rate", "max_precip_rate", "mm/hr"),
        ("Total rain fallen", "integrated_accumulated_precip", "kg"),
        (
            r"Min surface $\Delta \theta_v$",
            "surface_min_theta_v_perturbation",
            "K",
        ),
        ("Total water mass", "total_water_mass", "kg"),
        (
            r"$F_{up}$" + f" at z≈{actual_reference_height_m:.0f} m",
            "upward_mass_flux",
            "kg/s",
        ),
        ("Max cloud-top height", "max_cloud_top_height", "m"),
    ]

    n_panels = len(panel_specs)
    n_cols = 3
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axs = plt.subplots(
        ncols=n_cols,
        nrows=n_rows,
        figsize=(3.5 * n_cols, 3.0 * n_rows + 1),
        sharex=True,
        squeeze=False,
        layout="constrained",
    )

    simulation_time_minutes = ds["t_minutes"].values

    for panel_index, (panel_title, var_name, units_label) in enumerate(panel_specs):
        ax = axs.flatten()[panel_index]
        ax.plot(simulation_time_minutes, sentinel_series[var_name].values)
        ax.set_title(panel_title)
        ax.set_ylabel(units_label)

    # Hide unused panels in the grid
    for ax in axs.flatten()[n_panels:]:
        ax.set_axis_off()

    # Bottom row of populated panels gets the x-axis label
    for ax in axs[-1, :]:
        ax.set_xlabel("Simulation time (minutes)")

    return fig


for exp_name, cm1_ds in cm1_dss.items():
    fig = plot_sentinel_time_series(cm1_ds)
    fig.suptitle(exp_name)

## Extrema by height

In [ ]:
def plot_extended_extrema_banana(ds):
    """Time × height banana plots of column-wise extrema.

    Extends the existing `plot_extrema` panel:
    - splits W into max (updraft) and min (downdraft) so signs are visible
    - splits theta_e into max (warm/moist anomaly) and min (dry intrusion)
    - adds total-condensate and RH-perturbation panels
    Each panel shows the most extreme value over the (x, y) plane at
    each height level, evolving over time.
    """
    # Sum all available hydrometeor mixing ratios into one total-condensate field
    hydrometeor_mixing_ratio_vars = [
        v
        for v in ["RCP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP"]
        if v in ds.data_vars
    ]
    total_condensate_mixing_ratio = sum(ds[v] for v in hydrometeor_mixing_ratio_vars)
    # Horizontal wind speed magnitude at every grid point
    horizontal_wind_speed = np.sqrt(ds["UC"] ** 2 + ds["VC"] ** 2)

    # Helpers — column extrema with (z, time) orientation for pcolormesh
    def column_max(field):
        return field.max(["x", "y"]).transpose("z", "time")

    def column_min(field):
        return field.min(["x", "y"]).transpose("z", "time")

    # (title, dataarray, cmap)
    panel_specs = [
        ("Max updraft speed [m/s]", column_max(ds["WC"]), "viridis"),
        ("Min downdraft speed [m/s]", column_min(ds["WC"]), "viridis_r"),
        (r"Max $|\Delta \theta|$ [K]", column_max(np.abs(ds["THETA_bsr"])), "viridis"),
        ("Max horizontal wind [m/s]", column_max(horizontal_wind_speed), "viridis"),
        (r"Max $\Delta \theta_e$ [K]", column_max(ds["theta_e_bsr"]), "Reds"),
        (r"Min $\Delta \theta_e$ [K]", column_min(ds["theta_e_bsr"]), "Blues_r"),
        (
            "Max total condensate [kg/kg]",
            column_max(total_condensate_mixing_ratio),
            "viridis",
        ),
        (r"Max $\Delta$RH [-]", column_max(ds["RH_bsr"]), "viridis"),
    ]

    # Compute all panels together — dedupes chunk reads across panels
    panel_dataset = xr.Dataset(
        {f"panel_{i:02d}": spec[1] for i, spec in enumerate(panel_specs)}
    ).compute()

    n_panels = len(panel_specs)
    n_cols = 2
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axs = plt.subplots(
        ncols=n_cols,
        nrows=n_rows,
        figsize=(5 * n_cols, 2.3 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )

    simulation_time_minutes = ds["t_minutes"].values
    z_values = ds["z"].values

    for panel_index, (panel_title, _, cmap_name) in enumerate(panel_specs):
        ax = axs.flatten()[panel_index]
        field_values = panel_dataset[f"panel_{panel_index:02d}"].values
        pc = ax.pcolormesh(
            simulation_time_minutes,
            z_values,
            field_values,
            cmap=cmap_name,
        )
        fig.colorbar(pc, ax=ax)
        ax.set_title(panel_title)

    for ax in axs[-1, :]:
        ax.set_xlabel("Simulation time (minutes)")
    for ax in axs[:, 0]:
        ax.set_ylabel("z (m)")

    return fig


for exp_name, cm1_ds in cm1_dss.items():
    fig = plot_extended_extrema_banana(cm1_ds)
    fig.suptitle(exp_name)

## Means by height

In [ ]:
def plot_domain_mean_hovmoller(ds):
    """Time × height of the *domain-mean* profile — drift checks.

    Domain extrema (Section 2) hide problems that affect the *average*
    state without producing local outliers. Mean-profile Hovmöllers
    catch spurious heating, persistent drying, mass-flux drift over
    time, etc. — anything that shows up as a slow change in the bulk
    column rather than a local spike.
    """
    # Sum all available hydrometeor mixing ratios into one total-condensate field
    hydrometeor_mixing_ratio_vars = [
        v
        for v in ["RCP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP"]
        if v in ds.data_vars
    ]
    total_condensate_mixing_ratio = sum(ds[v] for v in hydrometeor_mixing_ratio_vars)

    # Domain-mean theta perturbation profile vs time (K)
    mean_theta_perturbation = ds["THETA_bsr"].mean(["x", "y"]).transpose("z", "time")

    # Domain-mean r_v perturbation profile vs time (kg/kg), relative to
    # the *initial* domain-mean profile (since RV has no built-in _bsr)
    initial_mean_rv_profile = ds["RV"].isel(time=0).mean(["x", "y"])
    mean_rv_perturbation = (
        (ds["RV"] - initial_mean_rv_profile).mean(["x", "y"]).transpose("z", "time")
    )

    # Domain-mean total condensate (kg/kg) — where the storm is putting mass aloft
    mean_total_condensate = total_condensate_mixing_ratio.mean(["x", "y"]).transpose(
        "z", "time"
    )

    # Domain-mean upward mass flux ⟨rho w⟩|_{w>0} vs height — overall convective profile.
    # Conditioning on w>0 separates the upward branch from the compensating subsidence.
    positive_w_indicator = (ds["WC"] > 0).astype(float)
    upward_mass_flux_density = ds["DN0"] * ds["WC"] * positive_w_indicator
    mean_upward_mass_flux = upward_mass_flux_density.mean(["x", "y"]).transpose(
        "z", "time"
    )

    # (title, dataarray, cmap, normalization_kind, colorbar_label)
    panel_specs = [
        (
            r"$\langle \Delta \theta \rangle$",
            mean_theta_perturbation,
            "RdBu_r",
            "diverging",
            "K",
        ),
        (
            r"$\langle \Delta r_v \rangle$",
            mean_rv_perturbation,
            "BrBG",
            "diverging",
            "kg/kg",
        ),
        (
            r"$\langle$ total condensate $\rangle$",
            mean_total_condensate,
            "viridis",
            "sequential",
            "kg/kg",
        ),
        (
            r"$\langle \rho w \rangle |_{w>0}$",
            mean_upward_mass_flux,
            "viridis",
            "sequential",
            r"kg m$^{-2}$ s$^{-1}$",
        ),
    ]

    # Compute all four panels together so dask reads input chunks once
    panel_dataset = xr.Dataset(
        {f"panel_{i:02d}": spec[1] for i, spec in enumerate(panel_specs)}
    ).compute()

    fig, axs = plt.subplots(
        ncols=2,
        nrows=2,
        figsize=(10, 6),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )

    simulation_time_minutes = ds["t_minutes"].values
    z_values = ds["z"].values

    for panel_index, (
        panel_title,
        _,
        cmap_name,
        normalization_kind,
        colorbar_label,
    ) in enumerate(panel_specs):
        ax = axs.flatten()[panel_index]
        field_values = panel_dataset[f"panel_{panel_index:02d}"].values
        if normalization_kind == "diverging":
            # Symmetric color scale around zero for sign-sensitive quantities
            field_max_absolute = float(np.nanmax(np.abs(field_values))) or 1.0
            norm_obj = colors.TwoSlopeNorm(
                vcenter=0,
                vmin=-field_max_absolute,
                vmax=field_max_absolute,
            )
            pc = ax.pcolormesh(
                simulation_time_minutes,
                z_values,
                field_values,
                cmap=cmap_name,
                norm=norm_obj,
            )
        else:
            pc = ax.pcolormesh(
                simulation_time_minutes,
                z_values,
                field_values,
                cmap=cmap_name,
            )
        fig.colorbar(pc, ax=ax, label=colorbar_label)
        ax.set_title(panel_title)

    for ax in axs[-1, :]:
        ax.set_xlabel("Simulation time (minutes)")
    for ax in axs[:, 0]:
        ax.set_ylabel("z (m)")

    return fig


for exp_name, cm1_ds in cm1_dss.items():
    fig = plot_domain_mean_hovmoller(cm1_ds)
    fig.suptitle(exp_name)

## Plan views

In [ ]:
def plot_peak_updraft_plan_views(ds, mid_level_reference_height_m=5000):
    """6-panel plan-view snapshot at the time of peak domain-wide updraft.

    Picks the single most-vigorous timestep in the run and shows the
    storm's spatial footprint at that moment: accumulated precip,
    surface cold pool, column-max W, cloud-top structure, integrated
    condensate (cloud water path proxy), and W at a chosen mid-level.
    """
    # Pick the timestep at which the domain-wide max W is largest
    domain_max_w_over_time = ds["WC"].max(["x", "y", "z"]).compute()
    peak_w_time_index = int(domain_max_w_over_time.argmax().values)
    snapshot_ds = ds.isel(time=peak_w_time_index)
    snapshot_time_minutes = float(snapshot_ds["t_minutes"].values)

    # Sum all available hydrometeor mixing ratios at the snapshot time
    hydrometeor_mixing_ratio_vars = [
        v
        for v in ["RCP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP"]
        if v in snapshot_ds.data_vars
    ]
    total_condensate_snapshot = sum(
        snapshot_ds[v] for v in hydrometeor_mixing_ratio_vars
    )

    # --- Per-panel 2D fields (all have dims y, x at this point) ---

    # 1. Accumulated precipitation up through the snapshot time
    accumulated_precip_2d = snapshot_ds["ACCPR"]

    # 2. Surface theta_v perturbation — cold pool footprint at lowest level.
    # `isel(z=0)` keeps z as a scalar coord (= lowest z-level value); drop it
    # so this panel can be combined into a Dataset with panels 1/3/4/5 (no z)
    # and panel 6 (different scalar z).
    surface_theta_v_perturbation_2d = (
        snapshot_ds["theta_v_bsr"].isel(z=0).drop_vars("z")
    )

    # 3. Column-max W — locates the most intense updraft anywhere in the column
    column_max_w_2d = snapshot_ds["WC"].max("z")

    # 4. Cloud-top height — highest z where total condensate exceeds threshold
    cloud_condensate_threshold_kg_per_kg = 1e-5
    cloud_mask_snapshot = (
        total_condensate_snapshot > cloud_condensate_threshold_kg_per_kg
    )
    cloudy_heights_snapshot = snapshot_ds["z"] * cloud_mask_snapshot
    cloud_top_height_2d = cloudy_heights_snapshot.max("z")

    # 5. Vertically integrated condensate — cloud water path proxy
    condensate_mass_density_snapshot = snapshot_ds["DN0"] * total_condensate_snapshot
    vertically_integrated_condensate_2d = condensate_mass_density_snapshot.integrate(
        "z"
    )

    # 6. W at a chosen mid-level — diagnoses anvil-base vertical motion.
    # `sel(z=..., method="nearest")` keeps z as a scalar coord; read the actual
    # selected value for the title, then drop the coord for the same Dataset-
    # combination reason as panel 2 above.
    w_at_mid_level_with_z_coord = snapshot_ds["WC"].sel(
        z=mid_level_reference_height_m, method="nearest"
    )
    actual_mid_level_z_m = float(w_at_mid_level_with_z_coord["z"].values)
    w_at_mid_level_2d = w_at_mid_level_with_z_coord.drop_vars("z")

    panel_specs = [
        (
            r"Accumulated rain (kg/m$^2$)",
            accumulated_precip_2d,
            "viridis",
            "sequential",
        ),
        (
            r"Surface $\Delta \theta_v$ (K)",
            surface_theta_v_perturbation_2d,
            "RdBu_r",
            "diverging",
        ),
        ("Column max updraft speed (m/s)", column_max_w_2d, "viridis", "sequential"),
        ("Cloud-top height (m)", cloud_top_height_2d, "viridis", "sequential"),
        (
            r"Condensate water path (kg/m$^2$)",
            vertically_integrated_condensate_2d,
            "viridis",
            "sequential",
        ),
        (
            r"$W$" + f" at z≈{actual_mid_level_z_m:.0f} m (m/s)",
            w_at_mid_level_2d,
            "RdBu_r",
            "diverging",
        ),
    ]

    # Compute all panels together — dask reads input chunks once
    panel_dataset = xr.Dataset(
        {f"panel_{i:02d}": spec[1] for i, spec in enumerate(panel_specs)}
    ).compute()

    fig, axs = plt.subplots(
        ncols=3,
        nrows=2,
        figsize=(12, 7),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )
    fig.suptitle(f"Peak-W snapshot at t = {snapshot_time_minutes:.1f} min")

    # Plot in km for readability
    x_km = snapshot_ds["x"].values / 1000
    y_km = snapshot_ds["y"].values / 1000

    for panel_index, (panel_title, _, cmap_name, normalization_kind) in enumerate(
        panel_specs
    ):
        ax = axs.flatten()[panel_index]
        # Make sure dim order is (y, x) for pcolormesh
        field_2d = panel_dataset[f"panel_{panel_index:02d}"].transpose("y", "x")
        if normalization_kind == "diverging":
            # Symmetric color scale around zero so sign is visually clear
            field_max_absolute = float(np.nanmax(np.abs(field_2d.values))) or 1.0
            norm_obj = colors.TwoSlopeNorm(
                vcenter=0,
                vmin=-field_max_absolute,
                vmax=field_max_absolute,
            )
            pc = ax.pcolormesh(
                x_km,
                y_km,
                field_2d.values,
                cmap=cmap_name,
                norm=norm_obj,
            )
        else:
            pc = ax.pcolormesh(x_km, y_km, field_2d.values, cmap=cmap_name)
        fig.colorbar(pc, ax=ax)
        ax.set_title(panel_title)
        ax.set_aspect("equal")

    for ax in axs[-1, :]:
        ax.set_xlabel("x (km)")
    for ax in axs[:, 0]:
        ax.set_ylabel("y (km)")

    return fig


for exp_name, cm1_ds in cm1_dss.items():
    fig = plot_peak_updraft_plan_views(cm1_ds)

# Parcels

## $n^*$ throughout simulation

### From Eulerian output

In [ ]:
cm1_ds = fv(cm1_dss)
parcel_ds = fv(parcels_dss)

nstar_time_ixs = np.arange(0, 91, 90)
fig, axs = plt.subplots(
    nrows=len(nstar_time_ixs),
    ncols=3,
    figsize=(10, 4 * len(nstar_time_ixs)),
    layout="constrained",
    squeeze=False,
    # sharex="col",
    # sharey="col",
)

for t_ix_ix, t_ix in enumerate(nstar_time_ixs):

    cm1_ds = cm1_dss[exp].isel(time=t_ix)
    parcel_ds = parcels_dss[exp].isel(time=t_ix)
    # short_exp = NICE_EXPERIMENT_NAMES[exp]
    n_parcels = len(parcel_ds["parcel_ix"])

    # Bin parcel z's at a fine resolution to get a histogram of parcel counts
    parcel_counts_fine = (
        parcel_ds["x"]
        .groupby_bins(
            parcel_ds["z"], bins=np.arange(0, cm1_ds["z"].max().values.item(), 200)
        )
        .count()
    )

    # Extract the midpoint of each fine bin to use as a z coordinate for interpolation
    fine_bin_midpoints = np.array(
        [z_bin.mid for z_bin in parcel_counts_fine["z_bins"].values]
    )

    # Re-wrap counts as a DataArray indexed on z (instead of the interval index)
    # so we can interpolate to the CM1 z grid
    parcel_counts_on_fine_z = xr.DataArray(
        parcel_counts_fine.values,
        dims=["z"],
        coords={"z": fine_bin_midpoints},
    )

    # Interpolate the parcel counts onto the CM1 z grid
    cm1_fine_grid = cm1_ds.interp(z=fine_bin_midpoints)
    # cm1_z_grid = cm1_ds["z"].values
    # parcel_counts_on_cm1_z = parcel_counts_on_fine_z.interp(z=cm1_z_grid)

    # Compute the horizontally averaged dry density on the CM1 z grid
    dry_density = (cm1_fine_grid["air_density"] / (1 + cm1_fine_grid["RV"])).squeeze()
    dry_air_mass = dry_density.integrate(["x", "y", "z"])
    z_grid_heights = dry_density["z"].diff("z")
    dry_density = dry_density.integrate(["x", "y"]) * np.append(
        z_grid_heights, z_grid_heights[-1]
    )

    # Left subplot: parcel counts and dry density, both on the CM1 z grid
    ax = axs[t_ix_ix, 0]
    ax.plot(
        parcel_counts_on_fine_z.values,
        fine_bin_midpoints,
        label="Parcel counts",
        color="C0",
    )
    ax.set_xlabel("Parcel counts per CM1 level", color="C0")
    ax.tick_params(axis="x", labelcolor="C0")
    ax.set_ylabel("z [m]")

    ax_twin = ax.twiny()
    ax_twin.plot(
        dry_density.values, fine_bin_midpoints, label="Dry density", color="C1"
    )
    ax_twin.set_xlabel("Dry density [kg m$^{-3}$]", color="C1")
    ax_twin.tick_params(axis="x", labelcolor="C1")

    ax.set_title(f"{exp}\n" + r"$N_p$ and $\rho_a$")

    ax = axs[t_ix_ix, 1]
    ax.plot(
        parcel_counts_on_fine_z.values / dry_density.values,
        fine_bin_midpoints,
    )
    ax.set_xlabel(r"Parcel counts / dry density [m$^3$ kg$^{-1}$]")
    ax.set_ylabel("z [m]")
    ax.set_title(f"{exp}\n" + r"$n^{*}$")

    mean_nstar = (n_parcels / dry_air_mass).values.item()
    ax.set_xlim([0.5 * mean_nstar, 1.5 * mean_nstar])

    # Right subplot: parcel counts normalized by dry density, both on CM1 z grid
    ax = axs[t_ix_ix, 2]
    ax.plot(
        parcel_counts_on_fine_z.values / dry_density.values / n_parcels,
        fine_bin_midpoints,
    )
    ax.set_xlabel(r"Parcel counts / dry density [m$^3$ kg$^{-1}$]")
    ax.set_ylabel("z [m]")
    ax.set_title(f"{exp}\n" + r"$n^{*}_{p}/N$")
    ax.set_xlim([0.5 * mean_nstar / n_parcels, 1.5 * mean_nstar / n_parcels])

    print(f"t={t_ix} minutes")
    print(f"One parcel = {(dry_air_mass / n_parcels).values.item():.2e} kg")
    print(r"Mean $n^*$=" + f"{mean_nstar:.2e}")
    print("")

ct.plotting.share_axes(axs[:, 1])
ct.plotting.share_axes(axs[:, 2])

### From tracked $\rho$ value

In [ ]:
cm1_ds = fv(cm1_dss)
parcel_ds = fv(parcels_dss)

nstar_time_ixs = np.arange(0, 91, 90)
fig, axs = plt.subplots(
    nrows=len(nstar_time_ixs),
    ncols=3,
    figsize=(10, 4 * len(nstar_time_ixs)),
    layout="constrained",
    squeeze=False,
    # sharex="col",
    # sharey="col",
)

for t_ix_ix, t_ix in enumerate(nstar_time_ixs):

    parcel_ds = parcels_dss[exp].isel(time=t_ix)
    n_parcels = len(parcel_ds["parcel_ix"])

    z_bins = np.arange(0, 18_000, 25)
    parcel_counts = (
        parcel_ds["x"]
        .groupby_bins(parcel_ds["z"], bins=z_bins, labels=z_bins[1:])
        .count()
    )
    mass_by_layer = (
        (parcel_ds["air_density"] / (1 + parcel_ds["RV"]))
        .groupby_bins(parcel_ds["z"], bins=z_bins, labels=z_bins[1:])
        .mean()
    )

    ax = axs[t_ix_ix, 1]
    ax.plot(
        parcel_counts.values / mass_by_layer.values,
        z_bins[1:],
    )
    ax.set_xlabel(r"Parcel counts / dry density [m$^3$ kg$^{-1}$]")
    ax.set_ylabel("z [m]")
    ax.set_title(f"{exp}\n" + r"$n^{*}$")

    # mean_nstar = (n_parcels / dry_air_mass).values.item()
    # ax.set_xlim([0.5 * mean_nstar, 1.5 * mean_nstar])

    # print(f"t={t_ix} minutes")
    # print(f"One parcel = {(dry_air_mass / n_parcels).values.item():.2e} kg")
    # print(r"Mean $n^*$=" + f"{mean_nstar:.2e}")
    # print("")

ct.plotting.share_axes(axs[:, 1])
ct.plotting.share_axes(axs[:, 2])

## Exploratory animation

In [ ]:
reload(sv.scene)
reload(sv)

plotter = sv.initialize_plotter()
# plotter.camera_position = [
#     (12851.915882752213, 12823.498565119762, 12661.16043770632),
#     (5873.05758755944, 5844.640269927004, 5682.302142513567),
#     (0.0, 0.0, 1.0),
# ]

# plotter.camera_set = True
scene = sv.Scene(_plotter=plotter, force_bounds=True)

parcels_ds = fv(small_parcels_dss)

scene_times = parcels_ds["time"][slice(None, None, 1)]

# sv.plot_gridded(
#     ds=rams_dss[exp].sel(time=scene_times),
#     contours={
#         "WC": dict(isosurfaces=[1, 2, 3, 4, 5], cmap="Greens", opacity=0.2),
#         # "R_condensate": dict(isosurfaces=[0.002, 0.004, 0.006, 0.008], cmap="Blues", opacity=0.2),
#     },
#     scene=scene,
#     show=False,
# )

sv.plot_trajectories(
    ds=parcels_ds.sel(time=scene_times).transpose("parcel_ix", "time"),
    scene=scene,
    color="#919191",
    show=False,
)


ANIMATE = False

if not ANIMATE:
    scene.show()
else:
    gif_path = Path("./figures/example_thermal_tracks.gif").resolve()
    gif_path.parent.mkdir(exist_ok=True)
    scene.animate(
        path=gif_path,
        fps=sv.animation.FPS(total_wall_seconds=8, raise_if_not_evenly_spaced=False),
    )

    # Display the animation
    display(Image(filename=gif_path))

# Entrainment

## Total entrainment vs time

In [ ]:
time_stride = slice(None, None, 1)

e_integral = e.isel(time=time_stride).integrate(["x", "y", "z"]).compute()
d_integral = d.isel(time=time_stride).integrate(["x", "y", "z"]).compute()

# Dividing by 1+qv gives dry air density
active_ds = cm1_ds.where(cm1_ds["active"], other=0)
active_mass = (
    (active_ds["air_density"] / (1 + active_ds["RV"]))
    .integrate(["x", "y", "z"])
    .compute()
)

In [ ]:
for exp, cm1_ds in cm1_dss.items():

    fig, axs = plt.subplots(ncols=2, figsize=(8, 3), layout="constrained")

    ax = axs[0]
    ax.plot(
        cm1_ds["t_minutes"].isel(time=time_stride),
        e_integral,
        label="Entrainment",
    )
    ax.plot(
        cm1_ds["t_minutes"].isel(time=time_stride),
        d_integral,
        label="Detrainment",
    )
    ax.set_xlabel("Simulation time (minutes)")
    ax.set_ylabel("Mass (kg)")
    ax.set_title("Entrainment and detrainment\nfrom Lagrangian parcels")
    ct.plotting.clean_legend(ax, frameon=False)

    # Should also match the updraft mass, to a scaling factor
    ax = axs[1]
    ax.plot(
        cm1_ds["t_minutes"].isel(time=time_stride),
        active_mass,
        label="Active mass",
    )
    twin_ax = ax.twinx()
    twin_ax.plot(
        cm1_ds["t_minutes"].isel(time=time_stride),
        parcels_dss[exp]["active"].sum("parcel_ix"),
        label="Active parcels",
        color="C1",
    )
    twin_ax.plot(
        cm1_ds["t_minutes"].isel(time=time_stride),
        # Reduce a bit so you can see it under the other curve
        parcels_dss[exp]["active"].sum("parcel_ix") * 0.9,
        label="Active parcels",
        linestyle="dashed",
        color="C1",
    )
    ax.set_xlabel("Simulation time (minutes)")
    ax.set_ylabel("Mass (kg)")
    ax.set_title("Eulerian vs Lagrangian\nupdraft mass")
    ct.plotting.clean_legend(ax, frameon=False)

In [ ]:
fig, ax = plt.subplots()

parcel_ds = fv(parcels_dss)

ax.plot(parcel_ds["t_minutes"], parcel_ds["active"].sum("parcel_ix"), label="Active")
net_entrained = parcel_ds["entraining"].sum("parcel_ix") - parcel_ds["detraining"].sum(
    "parcel_ix"
)
ax.plot(parcel_ds["t_minutes"], net_entrained, label="Net entrained")
ax.plot(
    parcel_ds["t_minutes"],
    net_entrained.cumsum("time"),
    label=r"$\Sigma$Net entrained",
    linestyle="dashed",
)


ax.plot(
    parcel_ds["t_minutes"], parcel_ds["entraining"].sum("parcel_ix"), label="Entraining"
)

ax.plot(
    parcel_ds["t_minutes"], parcel_ds["detraining"].sum("parcel_ix"), label="detraining"
)

fig.legend()
# ct.plotting.clean_legend(ax, frameon=False)

## Height of entrainment and detrainment events

In [ ]:
def bin_to_cm1_grid(parcel_ds, cm1_ds, dims=["x", "y", "z"], weights=None):
    cm1_ds = cm1_ds.copy()
    # Build bin edges: prepend a leading 0, compute interior midpoints between
    # adjacent grid cell centers, then extrapolate one final edge past the last center.
    # np.append only accepts two arrays; use np.concatenate for three-way joins.
    bin_edges = {
        dim: np.concatenate([
            # Leading edge: use 0 as the lower boundary of the first bin
            np.array([0]),
            # Interior edges: midpoint between each pair of adjacent cell centers
            (cm1_ds[dim] - (cm1_ds[dim].diff(dim) / 2)).values,
            # Trailing edge: extrapolate one half-spacing past the last center
            np.array([
                cm1_ds[dim].values[-1]
                + (cm1_ds[dim].values[-1] - cm1_ds[dim].values[-2]) / 2
            ]),
        ])
        for dim in dims
    }
    # Per-bin widths along each dim. Length = number of bins (len(edges) - 1).
    bin_widths = {dim: np.diff(bin_edges[dim]) for dim in dims}
    # Per-bin (x, y, z) volume, broadcast to a 3D array so we can divide the
    # binned histogram by it below to convert mass counts to a mass density.
    binned = histogram(
        *(parcel_ds[dim] for dim in dims),
        bins=list(bin_edges.values()),
        dim=["parcel_ix"],
        weights=weights,
    ).rename({f"{dim}_bin": dim for dim in dims})
    return binned

In [ ]:
e_events = bin_to_cm1_grid(
    parcel_ds=fv(parcels_dss),
    cm1_ds=fv(cm1_dss),
    dims=["z"],
    weights=fv(parcels_dss)["entraining"],
).compute()

d_events = bin_to_cm1_grid(
    parcel_ds=fv(parcels_dss),
    cm1_ds=fv(cm1_dss),
    dims=["z"],
    weights=fv(parcels_dss)["detraining"],
).compute()

e_events

In [ ]:
fig, ax = plt.subplots()

ax.plot(e_events.sum("time").values, e_events["z"])

# Get the level of maximum entrainment
z_max_entrainment = e_events.sum("time").idxmax("z")
ax.axhline(z_max_entrainment, color="C1", linestyle="dashed")


In [ ]:
# Calculate cloud base


for exp, cm1_ds in cm1_dss.items():
    in_cloud = cm1_ds["RCP"] >= 1e-5
    in_cloud.sum(["x", "y"]).plot(y="z")

plt.gca().axhline(z_max_entrainment)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 3), layout="constrained")

for ix, var in enumerate([e_events, d_events]):
    axs[ix].pcolormesh(var["time"].values, var["z"].values, var.values.T)

### Updraft mass vs time

In [ ]:
fig, ax = plt.subplots()


for exp, cm1_ds in cm1_dss.items():
    m_vs_t = (
        cm1_ds.where(cm1_ds["active"], other=0)["air_density"]
        .integrate(["x", "y", "z"])
        .compute()
    )
    ax.plot(cm1_ds["t_minutes"], m_vs_t.values)

#### Parcel origins at max mass

In [ ]:
origins

In [ ]:
mask = parcel_ds.isel(time=22, drop=True)["active"].values
# active_indices = np.where(mask)[0]
# active_indices

In [ ]:
masked = parcel_ds.isel(parcel_ix=mask)
masked

In [ ]:
    present_at_max = parcel_ds.isel(
        parcel_ix=parcel_ds.isel(time=22, drop=True)["active"].squeeze().values
    )

In [ ]:
for exp, parcel_ds in parcels_dss.items():
    present_at_max = parcel_ds.isel(
        parcel_ix=np.where(
            parcel_ds.isel(time=22, drop=True)["active"].squeeze().values
        )[0]
    )
    origins = present_at_max.isel(time=0)

    xy_counts = bin_to_cm1_grid(
        parcel_ds=origins,
        cm1_ds=fv(cm1_dss),
        dims=["x", "y"],
        weights=None,
    ).compute()

    z_counts = bin_to_cm1_grid(
        parcel_ds=origins,
        cm1_ds=fv(cm1_dss),
        dims=["z"],
        weights=None,
    ).compute()

xy_counts

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 3), layout="constrained")

axs[0].pcolormesh(xy_counts["x"], xy_counts["y"], xy_counts.values)

axs[1].plot(z_counts.values, z_counts["z"].values)